In [30]:
import pandas as pd
import sqlite3
import sentencepiece as spm
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MultiLabelBinarizer
torch.manual_seed(112358)

In [2]:
conn=sqlite3.connect("../data/archive.sqlite3")
cursor=conn.cursor()
df=pd.read_sql_query("SELECT * FROM posts,post_tags where posts.thread_id=post_tags.thread_id",conn)

In [3]:
df

,id,thread_id,user_id,created_at,thanks_count,nothanks_count,raw_html,processed_html,is_first_post,source,thread_id,tag
0,1,24681373,167643,1.647281e+09,0,0,<div></div>On a board the following six vector...,On a board the following six vectors are writt...,1,https://artofproblemsolving.com/community/p246...,24681373,Vectors
1,1,24681373,167643,1.647281e+09,0,0,<div></div>On a board the following six vector...,On a board the following six vectors are writt...,1,https://artofproblemsolving.com/community/p246...,24681373,geometry
2,2,24681347,167643,1.647280e+09,0,0,"<div></div>Let <img src=""//latex.artofproblems...",Let $\Gamma_1$ and $\Gamma_2$ be two circles e...,1,https://artofproblemsolving.com/community/p246...,24681347,geometry
3,3,24681324,167643,1.647280e+09,0,0,"<div></div>Let <img src=""//latex.artofproblems...",Let $P_1P_2...P_n$ be a regular $n$-gon in the...,1,https://artofproblemsolving.com/community/p246...,24681324,combinatorics
4,3,24681324,167643,1.647280e+09,0,0,"<div></div>Let <img src=""//latex.artofproblems...",Let $P_1P_2...P_n$ be a regular $n$-gon in the...,1,https://artofproblemsolving.com/community/p246...,24681324,geometry
...,...,...,...,...,...,...,...,...,...,...,...,...
1174634,345979,1556697,148231,1.393553e+09,2,0,"<div></div><a href=""http://www.artofproblemsol...",Kyiv Taras Shevchenko University Mechmat Compe...,0,https://artofproblemsolving.com/community/p155...,1556697,inequalities
1174635,345979,1556697,148231,1.393553e+09,2,0,"<div></div><a href=""http://www.artofproblemsol...",Kyiv Taras Shevchenko University Mechmat Compe...,0,https://artofproblemsolving.com/community/p155...,1556697,inequalities unsolved
1174636,345980,1556697,148231,1.398564e+09,2,0,<div></div>The following inequality is also tr...,The following inequality is also true.\nShow t...,0,https://artofproblemsolving.com/community/p155...,1556697,algebra
1174637,345980,1556697,148231,1.398564e+09,2,0,<div></div>The following inequality is also tr...,The following inequality is also true.\nShow t...,0,https://artofproblemsolving.com/community/p155...,1556697,inequalities


In [4]:
df=df[df.is_first_post==1]
df

,id,thread_id,user_id,created_at,thanks_count,nothanks_count,raw_html,processed_html,is_first_post,source,thread_id,tag
0,1,24681373,167643,1.647281e+09,0,0,<div></div>On a board the following six vector...,On a board the following six vectors are writt...,1,https://artofproblemsolving.com/community/p246...,24681373,Vectors
1,1,24681373,167643,1.647281e+09,0,0,<div></div>On a board the following six vector...,On a board the following six vectors are writt...,1,https://artofproblemsolving.com/community/p246...,24681373,geometry
2,2,24681347,167643,1.647280e+09,0,0,"<div></div>Let <img src=""//latex.artofproblems...",Let $\Gamma_1$ and $\Gamma_2$ be two circles e...,1,https://artofproblemsolving.com/community/p246...,24681347,geometry
3,3,24681324,167643,1.647280e+09,0,0,"<div></div>Let <img src=""//latex.artofproblems...",Let $P_1P_2...P_n$ be a regular $n$-gon in the...,1,https://artofproblemsolving.com/community/p246...,24681324,combinatorics
4,3,24681324,167643,1.647280e+09,0,0,"<div></div>Let <img src=""//latex.artofproblems...",Let $P_1P_2...P_n$ be a regular $n$-gon in the...,1,https://artofproblemsolving.com/community/p246...,24681324,geometry
...,...,...,...,...,...,...,...,...,...,...,...,...
1174622,345975,1556712,46787,1.247242e+09,2,0,"<div></div>For each nonzero integer <img src=""...",For each nonzero integer $ n$ find all functio...,1,https://artofproblemsolving.com/community/p155...,1556712,algebra unsolved
1174623,345975,1556712,46787,1.247242e+09,2,0,"<div></div>For each nonzero integer <img src=""...",For each nonzero integer $ n$ find all functio...,1,https://artofproblemsolving.com/community/p155...,1556712,function
1174627,345977,1556697,46787,1.247241e+09,1,0,<div></div>Show that for all integers <span st...,"Show that for all integers $ n \ge 2$, $ \sqrt...",1,https://artofproblemsolving.com/community/p155...,1556697,algebra
1174628,345977,1556697,46787,1.247241e+09,1,0,<div></div>Show that for all integers <span st...,"Show that for all integers $ n \ge 2$, $ \sqrt...",1,https://artofproblemsolving.com/community/p155...,1556697,inequalities


In [5]:
df.memory_usage(deep=True)

Index               1346408
id                  1346408
thread_id           1346408
user_id             1346408
created_at          1346408
thanks_count        1346408
nothanks_count      1346408
raw_html          323690920
processed_html     73736543
is_first_post       1346408
source             16741166
thread_id           1346408
tag                10120919
dtype: int64

In [6]:
tag=df["tag"].value_counts()
tag

tag
geometry           19077
algebra            11493
number theory      11148
combinatorics      10196
inequalities        4823
                   ...  
sets of numbers        1
Prove that             1
airlines               1
cities                 1
pco                    1
Name: count, Length: 3617, dtype: int64

In [7]:
# we 'll take the first 4
tag=tag[tag>10000]
tag

tag
geometry         19077
algebra          11493
number theory    11148
combinatorics    10196
Name: count, dtype: int64

In [8]:
df=df[df.tag.isin(tag.index)]

In [9]:
df

,id,thread_id,user_id,created_at,thanks_count,nothanks_count,raw_html,processed_html,is_first_post,source,thread_id,tag
1,1,24681373,167643,1.647281e+09,0,0,<div></div>On a board the following six vector...,On a board the following six vectors are writt...,1,https://artofproblemsolving.com/community/p246...,24681373,geometry
2,2,24681347,167643,1.647280e+09,0,0,"<div></div>Let <img src=""//latex.artofproblems...",Let $\Gamma_1$ and $\Gamma_2$ be two circles e...,1,https://artofproblemsolving.com/community/p246...,24681347,geometry
3,3,24681324,167643,1.647280e+09,0,0,"<div></div>Let <img src=""//latex.artofproblems...",Let $P_1P_2...P_n$ be a regular $n$-gon in the...,1,https://artofproblemsolving.com/community/p246...,24681324,combinatorics
4,3,24681324,167643,1.647280e+09,0,0,"<div></div>Let <img src=""//latex.artofproblems...",Let $P_1P_2...P_n$ be a regular $n$-gon in the...,1,https://artofproblemsolving.com/community/p246...,24681324,geometry
5,4,24681313,167643,1.647280e+09,0,0,"<div></div>Find, with proof, all functions <im...","Find, with proof, all functions $f : R - \{0\}...",1,https://artofproblemsolving.com/community/p246...,24681313,algebra
...,...,...,...,...,...,...,...,...,...,...,...,...
1174607,345970,1556755,46787,1.247244e+09,2,0,"<div></div>Consider a convex solid <img src=""/...",Consider a convex solid $ K$ in space and two ...,1,https://artofproblemsolving.com/community/p155...,1556755,geometry
1174610,345971,1556694,46787,1.247241e+09,1,0,<div></div>Determine the number of integers <i...,Determine the number of integers $ n$ with $ 1...,1,https://artofproblemsolving.com/community/p155...,1556694,number theory
1174615,345973,1556701,46787,1.247241e+09,2,0,<div></div>In a convex quadrilateral <span sty...,"In a convex quadrilateral $ ABCD$, let $ E$ be...",1,https://artofproblemsolving.com/community/p155...,1556701,geometry
1174621,345975,1556712,46787,1.247242e+09,2,0,"<div></div>For each nonzero integer <img src=""...",For each nonzero integer $ n$ find all functio...,1,https://artofproblemsolving.com/community/p155...,1556712,algebra


In [10]:
ds=df.loc[:, ~df.columns.duplicated()].groupby("thread_id",as_index=True).agg(
    text=("processed_html","first"),
    tag=("tag",list)
)
ds

,text,tag
thread_id,,
2,"Let $ABC$ be a triangle, and $M$ an interior p...",[geometry]
3,okay this one is from Prof. Mircea Lascu from ...,"[algebra, geometry]"
5,"If A,B are invertible and the set {Ak - Bk | k...",[algebra]
9,In a magic square $n \times n$ composed from t...,"[algebra, combinatorics]"
72,The lengths of the sides of a convex hexagon $...,[geometry]
...,...,...
36238654,"Let $a, b, c$ be the altitudes of triangle $A$...",[geometry]
36238689,Find all functions that satisfy the condition ...,[algebra]
36238706,On an $N \times N$ “chessboard” ($N \ge 3$) ea...,[combinatorics]


In [11]:
ds.memory_usage(deep=True)

Index      371648
text     19470208
tag       3729424
dtype: int64

In [12]:
sp = spm.SentencePieceProcessor(model_file="my_tokenizer.model")

In [27]:
def tokenize(text):
    return sp.encode(text,out_type=int)
X=ds["text"].apply(tokenize)
X

thread_id
2           [215, 3, 427, 7916, 81, 6, 332, 7929, 35, 3, 7...
3           [4808, 232, 149, 275, 29, 264, 387, 7924, 7937...
5           [320, 79, 7929, 7951, 101, 7886, 35, 9, 439, 4...
9           [622, 6, 7885, 471, 3, 7914, 8, 705, 45, 7916,...
72          [266, 2315, 31, 9, 927, 31, 6, 2166, 3058, 3, ...
                                  ...                        
36238654    [215, 3, 7912, 7929, 24, 7929, 18, 7916, 81, 9...
36238689    [1022, 170, 1892, 38, 1023, 9, 733, 98, 170, 5...
36238706    [1782, 22, 3, 7967, 8, 705, 147, 7916, 7909, 0...
36238733    [533, 29, 1689, 38, 156, 1611, 68, 2514, 256, ...
36238754    [622, 332, 3, 427, 48, 3, 311, 138, 217, 7958,...
Name: text, Length: 46456, dtype: object

In [31]:
mlb = MultiLabelBinarizer()
Y = mlb.fit_transform(ds["tag"])
print(mlb.classes_)

['algebra' 'combinatorics' 'geometry' 'number theory']


In [33]:
Y

array([[0, 0, 1, 0],
       [1, 0, 1, 0],
       [1, 0, 0, 0],
       ...,
       [0, 1, 0, 0],
       [0, 0, 0, 1],
       [0, 0, 1, 0]], shape=(46456, 4))

In [ ]:
class SeqDataset(Dataset):
    def __init__(self, X, Y):
        self.X = [torch.tensor(x, dtype=torch.long) for x in X]   
        self.Y = torch.tensor(Y, dtype=torch.float32)              

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

def collate_fn(batch):
    xs, ys = zip(*batch)
    x_lens = torch.tensor([len(x) for x in xs])
    x_padded = pad_sequence(xs, batch_first=True, padding_value=0)
    y_batch = torch.stack(ys)   # (batch, n_labels) — không cần pad
    return x_padded, y_batch, x_lens
dataset = SeqDataset(X, Y)
loader = DataLoader(dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)

In [39]:
x,y=next(iter(loader))

ValueError: too many values to unpack (expected 2)

In [38]:
x

(tensor([[  79, 1628,  439,  ...,    0,    0,    0],
         [2415,    6,  332,  ...,    0,    0,    0],
         [1022,    9, 1762,  ...,    0,    0,    0],
         ...,
         [ 635,   38,    9,  ...,    0,    0,    0],
         [ 512,  101, 7657,  ...,    0,    0,    0],
         [2421,    9, 1762,  ...,    0,    0,    0]]),
 tensor([[0., 1., 1., 0.],
         [0., 0., 1., 0.],
         [1., 0., 0., 0.],
         [0., 1., 0., 0.],
         [0., 0., 0., 1.],
         [0., 1., 0., 0.],
         [0., 0., 0., 1.],
         [0., 0., 1., 0.],
         [0., 0., 0., 1.],
         [0., 0., 0., 1.],
         [0., 0., 1., 0.],
         [0., 0., 0., 1.],
         [0., 1., 0., 0.],
         [1., 0., 0., 0.],
         [1., 0., 0., 0.],
         [0., 1., 0., 0.],
         [0., 0., 1., 0.],
         [0., 0., 0., 1.],
         [0., 0., 1., 0.],
         [0., 0., 1., 0.],
         [0., 0., 1., 0.],
         [1., 0., 0., 0.],
         [1., 0., 0., 0.],
         [0., 0., 1., 0.],
         [0., 1., 